In [89]:
import pandas as pd
import numpy as np

### Dagshub/Mlflow initialization ###

In [90]:
import dagshub
import mlflow
import mlflow.sklearn

dagshub.init(repo_owner='sbolk23', repo_name='House-Prices-Kaggle-Competition', mlflow=True)


Initialized MLflow to track repo "sbolk23/House-Prices-Kaggle-Competition"

Repository sbolk23/House-Prices-Kaggle-Competition initialized!

In [91]:
PATH = '../data'
TARGET = 'SalePrice'

In [92]:
# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)

### Split Train/Test Data ###

In [93]:
df = pd.read_csv(PATH + '/train.csv')

In [94]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold

train_df, test_df = train_test_split(
    df, test_size=.2, shuffle=True, random_state=1337
)

print(f'train_df shape {train_df.shape}')
print(f'test_df shape {test_df.shape}')

train_df shape (1168, 81)
test_df shape (292, 81)


### Data Cleaning ###

### Remove Outliers ###

In [95]:
outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)]

train_df = train_df.drop(outliers.index)

X_train = train_df.drop(columns=['SalePrice'])
y_train = train_df['SalePrice']

X_test = test_df.drop(columns=['SalePrice'])
y_test = test_df['SalePrice']

print(f'X_train shape {X_train.shape}')
print(f'X_test shape {X_test.shape}')

X_train shape (1166, 80)
X_test shape (292, 80)


### Categorical Imputing ###

In [96]:
# we first impute all missing values in categorical and numeric features.

# These categorical columns may have 'NaN' values in them which means missing.
# data_description.txt did not mention 'Nan' was intended.

cat_impute_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'ExterQual',
    'ExterCond',
    'Heating',
    'HeatingQC',
    'CentralAir',
    'Electrical',
    'KitchenQual',
    'Functional',
    'PavedDrive',
    'SaleType',
    'SaleCondition',
]

### Numeric Imputing ###

In [97]:
# We impute all numeric columns because data_description.txt did not mention any numeric feature with intended 'NaN' value.

# num_impute_cols = [
#     'LotFrontage',
#     'GarageYrBlt',
#     'MasVnrArea',
# ]

num_impute_cols = list(X_train.select_dtypes(exclude='object').columns)
num_impute_cols.remove('Id')
num_impute_cols.remove('MSSubClass')

### Categorical One-Hot-Encoded Columns ###

In [98]:
# Here we list features to be one-hot-encoded including 'NaN's because 'NaN' here does not mean missing.
# 'MasVnrType' was interesting column here.

cat_ohe_cols = [
    'MSZoning',
    'Street',
    'Alley',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'MasVnrType',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'GarageType',
    'PavedDrive',
    'MiscFeature',
    'SaleType',
    'SaleCondition',
]

cat_ohe_missing_cols = [
    'MSZoning',
    'Street',
    'LotShape',
    'LandContour',
    'Utilities',
    'LotConfig',
    'LandSlope',
    'Neighborhood',
    'Condition1',
    'Condition2',
    'BldgType',
    'HouseStyle',
    'RoofStyle',
    'RoofMatl',
    'Exterior1st',
    'Exterior2nd',
    'Foundation',
    'Heating',
    'CentralAir',
    'Electrical',
    'PavedDrive',
    'SaleType',
    'SaleCondition'
]

cat_ohe_absent_cols = [
    'Alley',
    'MasVnrType',
    'GarageType',
    'MiscFeature'
]

### Ordinal Columns ###

In [99]:
# Some categorical columns can may be ordered, replace them with numbers.

cat_ord_cols = [
    'ExterQual',
    'ExterCond',
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'HeatingQC',
    'KitchenQual',
    'Functional',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

cat_ord_missing_cols = [
    'ExterQual',
    'ExterCond',
    'HeatingQC',
    'KitchenQual',
    'Functional',
]

cat_ord_absent_cols = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
    'FireplaceQu',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
    'PoolQC',
    'Fence',
]

exter_qual_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
exter_cond_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_qual_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_cond_order         = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_exposure_order     = ['NA', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_type_1_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
bsmt_fin_type_2_order   = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
heating_qc_order        = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
kitchen_qual_order      = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
functional_order        = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
fireplace_qu_order      = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_finish_order     = ['NA', 'Unf', 'RFn', 'Fin']
garage_qual_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
garage_cond_order       = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
pool_qc_order           = ['NA', 'Fa', 'TA', 'Gd', 'Ex']
fence_order             = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

cat_ord_missing_categories = [
    exter_qual_order,
    exter_cond_order,
    heating_qc_order,
    kitchen_qual_order,
    functional_order,
]

cat_ord_absent_categories = [
    bsmt_qual_order,
    bsmt_cond_order,
    bsmt_exposure_order,
    bsmt_fin_type_1_order,
    bsmt_fin_type_2_order,
    fireplace_qu_order,
    garage_finish_order,
    garage_qual_order,
    garage_cond_order,
    pool_qc_order,
    fence_order,
]

### Numeric One-Hot-Encoded Columns ###

In [100]:
# This feature has numeric type which is misleading.

num_ohe_cols = [
    'MSSubClass',
]

### Irrelevant Columns ###

In [101]:
# This col is not informative.

irrelevant_cols = [
    'Id'
]

### Data Cleanup Pipeline ###

In [102]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# scaling every column after preprocessing decreased accuracy.
# scaling only numeric columns is sufficient.

num_impute_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

num_ohe_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

cat_ord_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=cat_ord_missing_categories)),
])

cat_ord_absent_pipeline = Pipeline([
    ('nan_impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='NA')),
    ('ordinal',     OrdinalEncoder(categories=cat_ord_absent_categories)),
])

cat_ohe_missing_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

cat_ohe_absent_pipeline = Pipeline([
    ('impute',  SimpleImputer(strategy='constant', missing_values=np.nan, fill_value='Missing')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num_impute',       num_impute_pipeline,       num_impute_cols),
        ('num_ohe',          num_ohe_pipeline,          num_ohe_cols),
        ('cat_ord_absent',   cat_ord_absent_pipeline,   cat_ord_absent_cols),
        ('cat_ord_missing',  cat_ord_missing_pipeline,  cat_ord_missing_cols),
        ('cat_ohe_absent',   cat_ohe_absent_pipeline,   cat_ohe_absent_cols),
        ('cat_ohe_missing',  cat_ohe_missing_pipeline,  cat_ohe_missing_cols),
        ('drop',            'drop',                     irrelevant_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

preprocessor.set_output(transform='pandas')

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

### Custom Scorer ###

In [103]:
from sklearn.metrics import make_scorer, root_mean_squared_log_error

def safe_rmsle(y_true, y_pred):
    y_pred_pos = np.maximum(y_pred, 0)
    return root_mean_squared_log_error(y_true, y_pred_pos)

safe_rmsle_scorer = make_scorer(safe_rmsle, greater_is_better=False)

### Full Pipeline (Linear Regression) ###

In [61]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

final_model = TransformedTargetRegressor(
    regressor=pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean'],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },
]

### MLFlow Logging (Linear Regression) ###

In [62]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v2')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'LR__prep_v2__{has_log}__num_imp_{num_imp}__ord_imp__ohe__no_outliers'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        mlflow.log_param('outlier_min_threshold_grlivarea', 4000)
        mlflow.log_param('outlier_max_threshold_price', 300000)
        mlflow.log_param('outlier_ids', str(list(outliers['Id'].index)))

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


2026/04/11 14:08:34 INFO mlflow.tracking.fluent: Experiment with name 'linear_regression_prep_v2' does not exist. Creating a new experiment.


Logging parameters (LR__prep_v2__logY__num_imp_mean__ord_imp__ohe__no_outliers)
Logging preprocessing parameters (LR__prep_v2__logY__num_imp_mean__ord_imp__ohe__no_outliers)
Logging train and validation metrics (LR__prep_v2__logY__num_imp_mean__ord_imp__ohe__no_outliers)
Logging final model (LR__prep_v2__logY__num_imp_mean__ord_imp__ohe__no_outliers)


2026/04/11 14:10:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LR__prep_v2__logY__num_imp_mean__ord_imp__ohe__no_outliers at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/0e81939d5066404caf92a8bebbcfe9d2
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2


### Full Pipeline (Ridge) ###

In [115]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

final_model = TransformedTargetRegressor(
    regressor=pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean',],
        'regressor__model__alpha': [.001, .01, .1, 1, 5, 10, 20, 25, 50, 100, 1000],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },

    #   'regressor__model__alpha': np.linspace(1, 50, 300),
]

### MLFlow Logging (Ridge) ###

In [116]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    verbose=1,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v2')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'RIDGE__alpha_{alpha}__prep_v2__{has_log}__num_imp_{num_imp}__ord_imp__ohe__no_outliers'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        mlflow.log_param('outlier_min_threshold_grlivarea', 4000)
        mlflow.log_param('outlier_max_threshold_price', 300000)
        mlflow.log_param('outlier_ids', str(list(outliers['Id'].index)))


        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


Fitting 5 folds for each of 11 candidates, totalling 55 fits
Logging parameters (RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 16:26:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_0.001__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/a4fa2c36d847474facd28c2e54ccc53f
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2
Logging parameters (RIDGE__alpha_0.01__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_0.01__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_0.01__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_0.01__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 16:32:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_0.01__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/d397e8a2b8a4444bac3e2cedc2e67cc6
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2
Logging parameters (RIDGE__alpha_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_0.1__prep_v1__logY__num_imp_mean__ord_imp__ohe)


...

🏃 View run RIDGE__alpha_50.0__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/ff268480256a4790b03f2ec2a3b6d368
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2
Logging parameters (RIDGE__alpha_100.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_100.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_100.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_100.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 17:10:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_100.0__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/5703b09207d944d4aeac46ca68f4db17
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2
Logging parameters (RIDGE__alpha_1000.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging preprocessing parameters (RIDGE__alpha_1000.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging train and validation metrics (RIDGE__alpha_1000.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)
Logging final model (RIDGE__alpha_1000.0__prep_v1__logY__num_imp_mean__ord_imp__ohe)


2026/04/11 17:12:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RIDGE__alpha_1000.0__prep_v1__logY__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2/runs/93b5fee2b16045c9ab8c179896a960f6
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/2


### Full Pipeline (Lasso) ###

In [119]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.compose import TransformedTargetRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Lasso(max_iter=100000))
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = [
    {
        'regressor__preprocessor__num_impute__impute__strategy': ['mean',],
        'regressor__model__alpha': [.0001, .0002, .0003, .0004, .0005, .0006, .0007, .0008, .0009, .001, .01, .1, 1, 5, 10],
        'func': [np.log1p],
        'inverse_func': [np.expm1],
    },

    # 'regressor__model__alpha': np.logspace(-4, 1, 300),
]

### MLFlow Logging (Lasso) ###

In [41]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression_prep_v2')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'LASSO__alpha_{alpha}__prep_v2__{has_log}__num_imp_{num_imp}__ord_imp__ohe__no_outliers'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        mlflow.log_param('outlier_min_threshold_grlivarea', 4000)
        mlflow.log_param('outlier_max_threshold_price', 300000)
        mlflow.log_param('outlier_ids', str(list(outliers['Id'].index)))

        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


KeyboardInterrupt: 

### Final Pipeline (ElasticNet) ###

In [32]:
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.compose import TransformedTargetRegressor

final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', ElasticNet(max_iter=10000)),
])

final_model = TransformedTargetRegressor(
    regressor=final_pipeline,
)

param_grid = {
    'regressor__preprocessor__num_impute__impute__strategy': ['mean'],
    'regressor__model__alpha': [.0005, .0006, .001, .1, .5, 1],
    'regressor__model__l1_ratio': [.1, .5, .9, .95, .99],
    'func': [np.log1p],
    'inverse_func': [np.expm1],

    # 'regressor__model__alpha': np.logspace(-4, 1, 30),
    # 'regressor__model__l1_ratio': np.linspace(.05, .95, 30),
}

Fitting 5 folds for each of 48 candidates, totalling 240 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_func,param_inverse_func,param_regressor__model__alpha,param_regressor__model__l1_ratio,param_regressor__preprocessor__num_impute__impute__strategy,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score,split0_train_score,split1_train_score,split2_train_score,split3_train_score,split4_train_score,mean_train_score,std_train_score
0,0.535899,0.147639,0.170236,0.067274,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.1,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.143523,-0.133331,-0.202475,-0.127415,-0.146919,-0.150733,0.026797,20,-0.095320,-0.095635,-0.093499,-0.097639,-0.093409,-0.095100,0.001562
1,0.682396,0.299321,0.145215,0.040291,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.5,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.143503,-0.122829,-0.204950,-0.114473,-0.134942,-0.144139,0.031991,15,-0.099061,-0.100468,-0.097348,-0.101885,-0.097915,-0.099336,0.001663
2,0.429842,0.022604,0.122629,0.033517,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.6,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.143890,-0.121680,-0.205288,-0.113427,-0.133475,-0.143552,0.032554,13,-0.099463,-0.101246,-0.098017,-0.102576,-0.098640,-0.099988,0.001689
3,1.055572,0.266010,0.154899,0.041594,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.7,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.144470,-0.120559,-0.205884,-0.112845,-0.132390,-0.143230,0.033111,11,-0.099845,-0.101922,-0.098677,-0.103285,-0.099285,-0.100603,0.001729
4,0.636231,0.166752,0.225417,0.103860,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.8,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.145020,-0.119512,-0.206440,-0.112422,-0.131721,-0.143023,0.033594,9,-0.100086,-0.102392,-0.099207,-0.104036,-0.099811,-0.101106,0.001820
5,0.686255,0.212899,0.176837,0.071054,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0005,0.9,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.146140,-0.118432,-0.206966,-0.112019,-0.131316,-0.142975,0.034063,7,-0.100200,-0.102845,-0.099748,-0.104589,-0.100146,-0.101506,0.001894
6,0.617965,0.146353,0.171829,0.054623,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0060,0.1,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.134801,-0.130975,-0.206233,-0.115559,-0.135998,-0.144713,0.031615,17,-0.116534,-0.117642,-0.106302,-0.120688,-0.116931,-0.115619,0.004882
7,0.440741,0.153425,0.230611,0.041430,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0060,0.5,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.141731,-0.139280,-0.213858,-0.129348,-0.144512,-0.153746,0.030488,22,-0.136704,-0.136826,-0.122917,-0.139971,-0.137095,-0.134703,0.006015
8,0.476066,0.157652,0.145079,0.045640,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0060,0.6,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.142401,-0.139420,-0.213970,-0.130606,-0.145739,-0.154427,0.030193,25,-0.138942,-0.140015,-0.124482,-0.142355,-0.139404,-0.137040,0.006388
9,0.557998,0.155234,0.188501,0.081466,<ufunc 'log1p'>,<ufunc 'expm1'>,0.0060,0.7,mean,"{'func': <ufunc 'log1p'>, 'inverse_func': <ufu...",-0.143344,-0.139954,-0.214031,-0.131853,-0.146399,-0.155116,0.029855,27,-0.140540,-0.142235,-0.125865,-0.144082,-0.140759,-0.138696,0.006540


In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone

kf = KFold(n_splits=5, shuffle=True, random_state=1337)

grid_search_cv = GridSearchCV(
    final_model,
    param_grid,
    cv=kf,
    scoring={
        'neg_root_mean_squared_log_error': safe_rmsle_scorer,
        'neg_root_mean_squared_error': 'neg_root_mean_squared_error',
        'neg_mean_absolute_error': 'neg_mean_absolute_error',
        'r2': 'r2',
    },
    refit='neg_root_mean_squared_log_error',
    return_train_score=True,
    n_jobs=-1
)

grid_search_cv.fit(X_train, y_train)
cv_results_df = pd.DataFrame(grid_search_cv.cv_results_)

mlflow.set_experiment('linear_regression')

for i, row in cv_results_df.iterrows():
    row_dict = row.to_dict()

    params = row_dict['params']
    alpha = row_dict['param_regressor__model__alpha']
    l1_ratio = row_dict['param_regressor__model__l1_ratio']
    num_imp = row_dict['param_regressor__preprocessor__num_impute__impute__strategy']
    has_log = "logY" if row_dict['param_func'] is not None else "rawY"

    run_name = f'ELASTIC_NET__alpha_{alpha}__l1_ratio_{l1_ratio}__prep_v2__{has_log}__num_imp_{num_imp}__ord_imp__ohe__no_outliers'

    with mlflow.start_run(run_name=run_name):
        print(f'Logging parameters ({run_name})')
        for key, value in row_dict.items():
            if 'param_' in key:
                mlflow.log_param(key.replace('param_', ''), value)

        mlflow.log_param('outlier_min_threshold_grlivarea', 4000)
        mlflow.log_param('outlier_max_threshold_price', 300000)
        mlflow.log_param('outlier_ids', str(list(outliers['Id'].index)))


        print(f'Logging preprocessing parameters ({run_name})')
        mlflow.log_param('num_impute_cols',         str(num_impute_cols))
        mlflow.log_param('num_ohe_cols',            str(num_ohe_cols))
        mlflow.log_param('cat_ord_absent_cols',     str(cat_ord_absent_cols))
        mlflow.log_param('cat_ord_missing_cols',    str(cat_ord_missing_cols))
        mlflow.log_param('cat_ohe_absent_cols',     str(cat_ohe_absent_cols))
        mlflow.log_param('cat_ohe_missing_cols',    str(cat_ohe_missing_cols))
        mlflow.log_param('irrelevant_cols',         str(irrelevant_cols))

        print(f'Logging train and validation metrics ({run_name})')
        for key, value in row_dict.items():
            if 'rank' in key:
                continue

            if 'neg_root_mean_squared_log_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_log_error', 'rmsle').replace('test', 'validation'), abs(value))

            if 'neg_root_mean_squared_error' in key:
                mlflow.log_metric(
                    key.replace('neg_root_mean_squared_error', 'rmse').replace('test', 'validation'), abs(value))

            if 'neg_mean_absolute_error' in key:
                mlflow.log_metric(
                    key.replace('neg_mean_absolute_error', 'mae').replace('test', 'validation'), abs(value))

            if 'r2' in key:
                mlflow.log_metric(key.replace('test', 'validation'), (value))


        # Logging time metrics
        mlflow.log_metric('mean_fit_time',      row_dict['mean_fit_time'])
        mlflow.log_metric('std_fit_time',       row_dict['std_fit_time'])
        mlflow.log_metric('mean_score_time',    row_dict['mean_score_time'])
        mlflow.log_metric('std_score_time',     row_dict['std_score_time'])

        model = clone(final_model)
        model.set_params(**params)

        print(f'Logging final model ({run_name})')
        model.fit(X_train, y_train)

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name='model',
        )

        mlflow.set_tag('model_type', 'LinearRegression')
        mlflow.set_tag('model_id', model_info.model_id)


In [88]:
from sklearn.metrics import root_mean_squared_log_error

mlflow.set_experiment('linear_regression_prep_v1')
model_id = 'm-2b268c9183ee44cdb145164676abe59b'
model = mlflow.sklearn.load_model(f'models:/{model_id}')
model.fit(X_train, y_train)
# y_pred = model.predict(X_test)
# print(root_mean_squared_log_error(y_test, y_pred))


# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Lasso(alpha=.0007934096665797492))
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))
#
#
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', LinearRegression())
# ])
# final_pipeline.fit(X_train, y_train_t)
# y_pred = final_pipeline.predict(X_test)
# print(root_mean_squared_error(y_test_t, y_pred))
#
# y_pred = final_pipeline.predict(X_train)
# print(root_mean_squared_error(y_train_t, y_pred))

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...ter=100000))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_impute', ...), ('num_ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. 

In [189]:
final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Ridge(alpha=400))
])

cv_results = cross_validate(
    final_pipeline, X_train, y_train_t,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    return_train_score=True,
)

cv_results_df = pd.DataFrame(cv_results)

train_root_mean_squared_error = -cv_results_df.mean()['train_score']
validation_root_mean_squared_error = -cv_results_df.mean()['test_score']

print(f'train_rmse: {train_root_mean_squared_error}')
print(f'validation_rmse: {validation_root_mean_squared_error}')

train_rmse: 0.10310912196089887
validation_rmse: 0.13957445547139208


In [124]:
# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', ElasticNet(alpha=0.0007278953843983154, l1_ratio=0.7131578947368421))
# ])

# final_pipeline = Pipeline([
#     ('preprocessor', preprocessor_pipeline),
#     ('model', Ridge(alpha=9.846791241561434))
# ])

final_pipeline = Pipeline([
    ('preprocessor', preprocessor_pipeline),
    ('model', Lasso(alpha=0.0006348565216305217))
])

test = pd.read_csv(PATH + '/test.csv')
test_ids = test['Id']

# final_pipeline.fit(pd.concat([X_train, X_test], axis=0, ignore_index=True), pd.concat([y_train_t, y_test_t], axis=0, ignore_index=True))
final_pipeline.fit(X_train, y_train_t)
prices_log = final_pipeline.predict(test)
prices = np.expm1(prices_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': prices,
})

submission.to_csv('submission.csv', index=False)

### Final Test scores ###

In [61]:
from sklearn.metrics import root_mean_squared_error

runs_df = mlflow.search_runs(experiment_names=["linear_regression"])

for run_id in runs_df["run_id"]:
    models = mlflow.search_logged_models(
        filter_string=f"source_run_id = '{run_id}'",
        output_format="list",
    )

    model_id = models[0].model_id
    model = mlflow.sklearn.load_model(f"models:/{model_id}")
    y_pred = model.predict(X_test)

    test_rmse = root_mean_squared_error(y_test_t, y_pred)

    with mlflow.start_run(run_id=run_id):
        mlflow.log_metric('test_rmse', test_rmse)


🏃 View run LR__prep_v1__num_imp_median__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/f7155e7f2be3435e919d5aab2e1b0bf5
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1


🏃 View run LR__prep_v1__num_imp_mean__ord_imp__ohe at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1/runs/339189ffc2fd4096a73dc763dce0fffb
🧪 View experiment at: https://dagshub.com/sbolk23/House-Prices-Kaggle-Competition.mlflow/#/experiments/1
